In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
# # Databricks notebook source
# import dlt
# from pyspark.sql.functions import col, sum, count, avg, count_distinct, round, current_timestamp, lit, broadcast

# # Configuration for high-volume joins (110M rows)
# spark.conf.set("spark.sql.shuffle.partitions", "400")

# # --- DOMAIN 5: AUDIT LOGGING HELPER ---
# def add_audit_metadata(df):
#     """Adds traceability columns for governance requirements."""
#     return df.withColumn("ingestion_timestamp", current_timestamp()) \
#              .withColumn("processed_by_user", lit("dlt_pipeline_service"))

# # --- KPI 1: PRODUCT PERFORMANCE ---
# @dlt.table(
#     name="product_performance",
#     comment="Gold Layer: Aggregated Brand Performance from Sales and Product Dimensions",
#     table_properties={
#         "quality": "gold",
#         # "delta.autoOptimize.optimizeWrite": "true", 
#         # "pipelines.autoOptimize.zOrderCols": "brand"
#     }
# )
# def calculate_product_performance():
#     # Dynamic DLT references - this creates the arrows in your graph
#     fact_sales = dlt.read("fact_sales")
#     dim_products = dlt.read("dim_products_scd2")

#     # Broadcast join for the smaller dimension table against the large fact table
#     joined_df = fact_sales.join(broadcast(dim_products), on="product_id", how="inner")
    
#     kpi_df = (
#         joined_df.groupBy(dim_products["brand"])
#         .agg(
#             round(sum(fact_sales["price"]), 2).alias("actual_revenue"),
#             round(avg(dim_products["price"]), 2).alias("historical_catalog_price"),
#             count(fact_sales["product_id"]).alias("total_units_sold"),
#             count_distinct(fact_sales["user_session"]).alias("total_sessions")
#         )
#     )
    
#     return add_audit_metadata(kpi_df)



In [0]:
# # --- KPI 2: CUSTOMER JOURNEY METRICS ---
# @dlt.table(
#     name="customer_metrics",
#     comment="Gold Layer: Behavioral and session-level conversion behavior",
#     table_properties={"quality": "gold"}
# )
# def customer_journey_metrics():
#     # Read from the cleaned silver events
#     return add_audit_metadata(
#         dlt.read("events_cleaned")
#         .groupBy("user_id")
#         .agg(
#             count_distinct("user_session").alias("total_sessions"),
#             count(col("event_type")).alias("total_interactions"),
#             round(avg(col("price")), 2).alias("avg_interaction_value")
#         )
#     )

In [0]:
# Databricks notebook source
import dlt
from pyspark.sql.functions import col, sum, count, avg, count_distinct, round, current_timestamp, lit, broadcast

# Configuration for 110M row shuffles
spark.conf.set("spark.sql.shuffle.partitions", "400")

def add_audit_metadata(df):
    return df.withColumn("ingestion_timestamp", current_timestamp()) \
             .withColumn("processed_by_user", lit("dlt_pipeline_service"))

@dlt.table(
    name="product_performance",
    comment="Gold Layer: KPI from Sales and SCD2 Dimensions",
    table_properties={"quality": "gold"}
)
def calculate_product_performance():
    # DIRECT LOCAL REFERENCES ONLY
    # If arrows don't appear, check if fact_sales is defined in another notebook 
    # and if that notebook is added to the pipeline settings.
    fact_sales = dlt.read("fact_sales") 
    dim_products = dlt.read("dim_products_scd2")

    joined_df = fact_sales.join(broadcast(dim_products), on="product_id", how="inner")
    
    return add_audit_metadata(
        joined_df.groupBy(dim_products["brand"])
        .agg(
            round(sum(fact_sales["price"]), 2).alias("actual_revenue"),
            round(avg(dim_products["price"]), 2).alias("historical_catalog_price"),
            count(fact_sales["product_id"]).alias("total_units_sold"),
            count_distinct(fact_sales["user_session"]).alias("total_sessions")
        )
    )

@dlt.table(
    name="customer_metrics",
    comment="Gold Layer: behavioral conversion behavior",
    table_properties={"quality": "gold"}
)
def customer_journey_metrics():
    # DIRECT LOCAL REFERENCE
    return add_audit_metadata(
        dlt.read("events_cleaned")
        .groupBy("user_id")
        .agg(
            count_distinct("user_session").alias("total_sessions"),
            count(col("event_type")).alias("total_interactions"),
            round(avg(col("price")), 2).alias("avg_interaction_value")
        )
    )